In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [3]:
url = "https://www.sas.am/ru/catalog/molochnye_produkty/"

request = requests.get(url)
print(request.status_code)

200


In [4]:
soup = BeautifulSoup(request.text, "html.parser")

#the data we need is kept in sections
print(soup.select_one("section"))

<section class="product-slider">
<div class="container">
<div class="product-slider__header">
<h2 class="product-slider__title h2">Молоко</h2>
<a class="product-slider__link-more link" href="/ru/catalog/moloko/">Все товары</a>
</div>
<div class="product-slider__slider scrollbar-none flc js-product-slider">
<div class="product-slider__slide">
<div class="product-wrap js-product-wrap">
<div class="product js-product js-hover-dropdown-bk product--web-slider">
<a class="product__cover-link" href="/ru/catalog/moloko/7271/"></a>
<div class="product-tags hidden-sm">
<div class="product-tags__item flc" style="background-color: #0FA3F7">хит</div>
</div>
<div class="product__favorite-btn">
<button aria-label="Добавить в избранное" class="favorite-btn js-favorite-btn" data-id="7271" type="button">
<svg class="svg-icon favorite-btn__icon favorite-btn__icon--stroke">
<use xlink:href="#svg-icon-heart-fill-white"></use>
</svg>
<svg class="svg-icon favorite-btn__icon favorite-btn__icon--fill">
<use xl

In [5]:
product_sliders = soup.select("section.product-slider")

print(len(product_sliders))

11


In [6]:
data = []

#worked with product_sliders = soup.select and not with find_all, why?????
for product_slider in product_sliders:
  category_name = product_slider.select_one("h2.product-slider__title").get_text(strip=True)
  # data[category_name] = {}
  products = product_slider.select("div.product-slider__slide")
  for product in products:
    prod_name_el = product.select_one("div.product__name")
    prod_price_el = product.select_one("div.price__new")
    if prod_name_el:
      prod_name = prod_name_el.get_text(strip=True)
      if prod_price_el:
        #because there is price, currency in span. we take the first
        prod_price = product.select_one(".price__text").contents[0].strip()
      else:
        prod_price = "-1"

        print(prod_name)

      prod_price = int("".join(prod_price.split()))
      data.append({
          "Category": category_name,
          "Product name": prod_name,
          "Price": prod_price
      })

print(data)

[{'Category': 'Молоко', 'Product name': 'Молоко "Yeremyan Products" 750мл, жирность: 3.2%', 'Price': 990}, {'Category': 'Молоко', 'Product name': "Молоко ''Марианна'' 950мл, жирность: 3,2%", 'Price': 660}, {'Category': 'Молоко', 'Product name': 'Молоко  ультрапастеризованное "Марианна" 1л, жирность:3.2%', 'Price': 920}, {'Category': 'Молоко', 'Product name': "Молоко ''Parmalat'' 1л, жирность:3.5%", 'Price': 1500}, {'Category': 'Молоко', 'Product name': 'Молоко без лактозы "Parmalat" 1л, жирность: 3.5%', 'Price': 1600}, {'Category': 'Молоко', 'Product name': 'Молоко "Yeremyan Products" 250мл, жирность: 3.2%', 'Price': 390}, {'Category': 'Молоко', 'Product name': 'Молоко "Еремян Продактс" 750мл, жирность: 0.5%', 'Price': 920}, {'Category': 'Молоко', 'Product name': 'Молоко без лактозы "Parmalat Protein" 1л, жирность: 1.8%', 'Price': 1590}, {'Category': 'Молоко', 'Product name': 'Молоко  "Parmalat" 1л,  жирность: 1.8%', 'Price': 1690}, {'Category': 'Молоко', 'Product name': 'Молоко "Lacte

In [8]:
df = pd.DataFrame(data)
print(df.head())

df.to_csv(
    "../data/raw/dairy.csv",
    index=False
)

  Category                                       Product name  Price
0   Молоко   Молоко "Yeremyan Products" 750мл, жирность: 3.2%    990
1   Молоко          Молоко ''Марианна'' 950мл, жирность: 3,2%    660
2   Молоко  Молоко  ультрапастеризованное "Марианна" 1л, ж...    920
3   Молоко              Молоко ''Parmalat'' 1л, жирность:3.5%   1500
4   Молоко   Молоко без лактозы "Parmalat" 1л, жирность: 3.5%   1600


In [ ]:
from playwright.async_api import async_playwright

async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto("https://www.sas.am/en/")

        selector = "a.sa-categories__item"

        await page.locator(selector).first.wait_for()

        # print("Before click:", await page.locator(selector).count())

        button = page.locator("button.sa-categories__more-btn") 

        await button.click()

        await page.wait_for_function(
            """() => document.querySelectorAll(
                'a.sa-categories__item'
            ).length > 24"""
        )

        html = await page.content()

        await browser.close()

    return html

html = await main()

soup = BeautifulSoup(html, "html.parser")

#the first layer. choosing if it's dairy, meat, sas product etc.
category_list = soup.select("a.sa-categories__item")

In [17]:
from urllib.parse import urljoin

#for class mapping
nutrient_classes = {
    "bg-blue-light": "kcal",
    "bg-blue": "Proteins",
    "bg-green": "Fats",
    "bg-green-light": "Carbohydrates"
}

#for url parsing
base_url = "https://www.sas.am"

banned_categories = ["Tobacco Products", "Everything for Animals"]

data = []
for category_item in category_list:
    category_name = category_item.get_text(strip=True)

    if category_name in banned_categories:
        continue

    category_url = urljoin(base_url, category_item.get("href"))
    
    #enter category(dairy)
    category_request = requests.get(category_url)
    
    subcategory_soup = BeautifulSoup(category_request.text, "html.parser")
    
    subcategory_list = subcategory_soup.select("a.catalog-category__item")
    for subcategory_item in subcategory_list:
        subcategory_name = subcategory_item.select_one("div.catalog-category__name").get_text(strip=True)

        subcategory_url = urljoin(base_url, subcategory_item.get("href"))

        #enter subcategory(milk)
        subcategory_request = requests.get(subcategory_url)

        products_soup = BeautifulSoup(subcategory_request.text, "html.parser")

        #pagination
        while True:
            products_list = products_soup.select("a.product__cover-link")
            
            for products_item in products_list:
                product_url = urljoin(base_url, products_item.get("href"))

                #enter product(marianna 1l milk)
                product_request = requests.get(product_url)
        
                product_soup = BeautifulSoup(product_request.text, "html.parser")

                product_name = product_soup.select_one("h1.card__title").get_text(strip=True)
                product_price = product_soup.select_one("span.price__text").contents[0].strip()
                item = {"Category name": category_name, "Subcategory name": subcategory_name, "Product name": product_name, "Product price": product_price}
                
                details = product_soup.select("div.card__detail-item")
                for detail in details:
                    #if there is a promotion, it is a card__detail-item too, and doesn't have any subdiv
                    title = detail.select_one("div.card__detail-item-title")
                    if not title:
                        continue
                    product_detail = title.get_text(strip=True)
                    if product_detail == "Brand":
                        product_brand = detail.select_one("div.card__detail-item-value").get_text(strip=True)
                        break
                else:
                    product_brand = ""
                item["Brand"] = product_brand

                nutrition = {
                    "kcal": None,
                    "Proteins": None,
                    "Fats": None,
                    "Carbohydrates": None
                }

                headers = product_soup.select_one("div.composition")

                if headers:
                    nutrition_items = headers.select("div.composition__item-header")
                    for nutrition_item in nutrition_items:
                        classes = nutrition_item.get("class", [])
                        for css_class, nutrient in nutrient_classes.items():
                            if css_class in classes:
                                value = nutrition_item.find_next_sibling("div")

                                if value:
                                    nutrition[nutrient] = value.get_text(strip=True) or None

                                break

                item.update(nutrition)
                data.append(item)
                # print(item)


            #get pagination button
            next_button = products_soup.select_one("a.pagination__arrow--right")

            if next_button:
                pagination_url = urljoin(base_url, next_button["href"])
                pagination_request = requests.get(pagination_url)

                products_soup = BeautifulSoup(pagination_request.text, "html.parser")
            else:
                break

print(len(data))
        

KeyboardInterrupt: 

In [21]:
print(len(data))
df = pd.DataFrame(data)
df[df.duplicated()]
# print(df.head())

# df.to_csv(
#     "../data/raw/second.csv",
#     index=False
# )

10246


,Category name,Subcategory name,Product name,Product price,Brand,kcal,Proteins,Fats,Carbohydrates
307,Gluten free,"Flakes, oatmeal","Oat flakes ""Nordic Gluten Free"" 400g",1 790,Nordic,370,14,8,55
870,Sugar free products,"Porridge, muesli","Granola ""Akha"" 300g",1 630,AXA,None,None,None,None
940,Sugar free products,Chocolate stick,"Oat bar ""Rice up"" 70g",640,Rice Up,None,None,None,None
970,Sugar free products,Chocolate stick,"Oat bar ""Rice up"" 70g",640,Rice Up,None,None,None,None
971,Sugar free products,Chocolate stick,"Oat bar ""Rice up"" 70g",640,Rice Up,None,None,None,None
...,...,...,...,...,...,...,...,...,...
10197,Beverages,"Cocktails, energy drinks, low-alcohol beverages","Apple cider ""Dargett "" 0.33l",1 090,Dargett,None,None,None,None
10217,Beverages,"Cocktails, energy drinks, low-alcohol beverages","Apple cider ""Villa Armos"" 0.33l",1 570,Villa Armos,None,None,None,None
10220,Beverages,"Cocktails, energy drinks, low-alcohol beverages","Light alcoholic drink ""Santo Stefano Bianco"" 0...",2 350,Santo Stefano,None,None,None,None
10225,Beverages,"Cocktails, energy drinks, low-alcohol beverages","Energy carbonated drink ""Max Power"" 0.33l",350,Max Power,None,None,None,None
